In [1]:
# Import required modules (now from root directory)
import tools.vectorstore as vs
from tools.registry import create_registry

In [2]:
# Check registry status first
registry = create_registry()
stats = registry.get_stats()
print(f"Registry Status:")
print(f"  Total files: {stats['total_files']}")
print(f"  Total chunks: {stats['total_chunks']}")
print(f"  Last updated: {stats['last_updated']}")
print(f"  Files: {stats['files']}")
print()

Registry Status:
  Total files: 3
  Total chunks: 121
  Last updated: 2025-09-28T09:10:25.202976
  Files: ['docs/knowledge/Supplier_PRD_Sop.txt', 'docs/knowledge/csv_descriptions/otif_pull.md', 'docs/knowledge/csv_descriptions/OnTime_Data.md']



In [3]:
# Get vectorstore and test search
vectorstore = vs.get_vectorstore()
print(f"Vectorstore type: {type(vectorstore)}")

# Test basic search
docs = vectorstore.similarity_search("Search the CSV data for PO355426 in OTIF, I want to know the PRD", k=5,
                                     filter={"knowledge_path": "csv_descriptions"}
                                     )
print(f"\nSearch for 'PO' found {len(docs)} documents:")

# Display results
for i, doc in enumerate(docs):
    print(f"\n--- Document {i+1} ---")
    print(f"Content preview: {doc.page_content}...")
    print(f"Source: {doc.metadata.get('filename', 'Unknown')}")
    print(f"Knowledge path: {doc.metadata.get('knowledge_path', '')}")
    print(f"Full metadata: {doc.metadata}")

Vectorstore type: <class 'langchain_chroma.vectorstores.Chroma'>

Search for 'PO' found 5 documents:

--- Document 1 ---
Content preview: # otif_pull.csv...
Source: otif_pull.md
Knowledge path: csv_descriptions
Full metadata: {'source': 'docs/knowledge/csv_descriptions/otif_pull.md', 'knowledge_path': 'csv_descriptions', 'folder': 'docs/knowledge', 'filename': 'otif_pull.md'}

--- Document 2 ---
Content preview: # View first few rows
df.head()
```

## 🔍 Understanding the OTIF Structure

### What is OTIF?
**On-Time In-Full (OTIF)** measures two critical supply chain metrics:
- **On-Time**: Did the order arrive by the promised date?
- **In-Full**: Was the complete quantity delivered?

### Key Identifier Fields
```
Purchase Order Level
    ├── document number (PO number)
    ├── internal id (System ID)
    └── po_razin_id (Composite PO-SKU identifier)
        └── Line Level
            ├── line id (Line item identifier)
            └── item (Product SKU/RAZIN)
                └── Fulfillm

In [4]:
docs

[Document(id='8aab39c9-caf8-42fc-981a-c49fabded257', metadata={'source': 'docs/knowledge/csv_descriptions/otif_pull.md', 'knowledge_path': 'csv_descriptions', 'folder': 'docs/knowledge', 'filename': 'otif_pull.md'}, page_content='# otif_pull.csv'),
 Document(id='2a39172f-8e5a-4340-839f-13dbcdb16ecf', metadata={'filename': 'otif_pull.md', 'folder': 'docs/knowledge', 'source': 'docs/knowledge/csv_descriptions/otif_pull.md', 'knowledge_path': 'csv_descriptions'}, page_content='# View first few rows\ndf.head()\n```\n\n## 🔍 Understanding the OTIF Structure\n\n### What is OTIF?\n**On-Time In-Full (OTIF)** measures two critical supply chain metrics:\n- **On-Time**: Did the order arrive by the promised date?\n- **In-Full**: Was the complete quantity delivered?\n\n### Key Identifier Fields\n```\nPurchase Order Level\n    ├── document number (PO number)\n    ├── internal id (System ID)\n    └── po_razin_id (Composite PO-SKU identifier)\n        └── Line Level\n            ├── line id (Line item 

In [5]:
# Load the otif_pull.csv file
import pandas as pd
otif_df = pd.read_csv('docs/csvs/otif_pull.csv')
# Filter the data for the user query PO355426
result = otif_df[otif_df['document number'] == 'PO377368']

In [6]:
result

,Unnamed: 0,internal id,date created,document number,associated brands,id,name,supplier confirmation status,final status,memo (main),...,current status,sub status,days bucket,team,reporting status,reporting days bucket,l2 final status,final poc,final team,snapshot_ts
0,0,66928377,2025-08-13,PO377368,Prestee,NaN,"74964 JIANGSU PLUS INTERNATIONAL TRADING CO.,LTD",Confirmed,Pending Receipt,Razor_July_Cycle_2025,...,20. Pre Pickup Check,20a. QC Release Missing,On-Track,CN,05. Pickup + Shipping,NaN,QC scheduled,Young Cao,Compliance & QC,2025-09-23 14:45:50.495636+00:00
998,998,66928377,2025-08-13,PO377368,Prestee,NaN,"74964 JIANGSU PLUS INTERNATIONAL TRADING CO.,LTD",Confirmed,Pending Receipt,Razor_July_Cycle_2025,...,20. Pre Pickup Check,20a. QC Release Missing,On-Track,CN,05. Pickup + Shipping,NaN,QC scheduled,Young Cao,Compliance & QC,2025-09-23 14:45:50.495636+00:00
2322,2322,66928377,2025-08-13,PO377368,Prestee,NaN,"74964 JIANGSU PLUS INTERNATIONAL TRADING CO.,LTD",Confirmed,Pending Receipt,Razor_July_Cycle_2025,...,20. Pre Pickup Check,20a. QC Release Missing,On-Track,CN,05. Pickup + Shipping,NaN,QC scheduled,Young Cao,Compliance & QC,2025-09-23 14:45:50.495636+00:00


In [7]:
otif_df

,Unnamed: 0,internal id,date created,document number,associated brands,id,name,supplier confirmation status,final status,memo (main),...,current status,sub status,days bucket,team,reporting status,reporting days bucket,l2 final status,final poc,final team,snapshot_ts
0,0,66928377,2025-08-13,PO377368,Prestee,NaN,"74964 JIANGSU PLUS INTERNATIONAL TRADING CO.,LTD",Confirmed,Pending Receipt,Razor_July_Cycle_2025,...,20. Pre Pickup Check,20a. QC Release Missing,On-Track,CN,05. Pickup + Shipping,NaN,QC scheduled,Young Cao,Compliance & QC,2025-09-23 14:45:50.495636+00:00
1,1,67247335,2025-09-17,PO378507,Bolt Dropper,NaN,"73109 YOLINK FASTENER (JIAXING) CO., LTD",Rejected,Pending Receipt,Razor_August_Cycle_2025,...,02. Supplier Confirmation Pending,02c. Rejected,01-03,CN,01. Pre Production,NaN,No L2 Status,Paul Fong,CM,2025-09-23 14:45:50.495636+00:00
2,2,63648408,2025-04-14,PO372477,Kidzlane,NaN,"74634 Wah Shing Toys Co.,Ltd.",Confirmed,Partially Received,Q4_2025_Batch1,...,30. Stock Delivery Pending,30c. Appointment in Future,On-Track,CN,07. Arrived,NaN,No L2 Status,NaN,NaN,2025-09-23 14:45:50.495636+00:00
3,3,67247335,2025-09-17,PO378507,Bolt Dropper,NaN,"73109 YOLINK FASTENER (JIAXING) CO., LTD",Rejected,Pending Receipt,Razor_August_Cycle_2025,...,02. Supplier Confirmation Pending,02c. Rejected,01-03,CN,01. Pre Production,NaN,No L2 Status,Paul Fong,CM,2025-09-23 14:45:50.495636+00:00
4,4,63648408,2025-04-14,PO372477,Kidzlane,NaN,"74634 Wah Shing Toys Co.,Ltd.",Confirmed,Partially Received,Q4_2025_Batch1,...,30. Stock Delivery Pending,30c. Appointment in Future,On-Track,CN,07. Arrived,NaN,No L2 Status,NaN,NaN,2025-09-23 14:45:50.495636+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3139,3139,67247416,2025-09-17,PO378586,JTJ Sourcing,NaN,70370 GST ENTERPRISE LIMITED,Confirmed,Pending Receipt,Razor_August_Cycle_2025,...,06. Packaging Pending,06b. SCM Check Pending,01-03,CN,01. Pre Production,NaN,No L2 Status,Darren Fernandes,Packaging,2025-09-23 14:45:50.495636+00:00
3140,3140,67247418,2025-09-17,PO378588,Jewelkeeper,NaN,75146 TE Electronic Manufacturing Ltd.,Rejected,Pending Receipt,Razor_August_Cycle_2025,...,02. Supplier Confirmation Pending,02c. Rejected,01-03,CN,01. Pre Production,NaN,No L2 Status,Jeremy Lin,CM,2025-09-23 14:45:50.495636+00:00
3141,3141,65828912,2025-05-28,PO373879,Vdomus,NaN,71649 Shenzhen Hongdashun acrylic Products Co....,Confirmed,Pending Receipt,Consolidation_PO366387,...,A. Anti PO Line,A. Anti PO Line,On-Track,CN,00. Anti + Blocked,NaN,c. Never pickup,NaN,NaN,2025-09-23 14:45:50.495636+00:00
3142,3142,65568609,2025-05-12,PO373323,InstallGear,NaN,75242 KINGSUN NETWORK DEVICES VIETNAM COMPANY ...,Confirmed,Pending Receipt,Q4_2025_Batch2,...,28. Telex Release Pending,28a. In Transit - Supplier Pending,On-Track,CN,06. On Sea,NaN,Yellow2:Not Released_Pending Vendor release to...,Dede Wu,SM,2025-09-23 14:45:50.495636+00:00


In [8]:
targeted_docs = vectorstore.similarity_search(
                    "column names reference",
                    k=10,
                    filter={
                        "$and": [
                            {"section_type": {"$eq": "column_reference"}},
                            {"filename": {"$eq": "otif_pull.md"}}
                        ]
                    }
                )

In [10]:
targeted_docs

[Document(id='7a6bcd02-e8a6-489e-bc67-fd6b3e8c240f', metadata={'folder': 'docs/knowledge', 'knowledge_path': 'csv_descriptions', 'source': 'docs/knowledge/csv_descriptions/otif_pull.md', 'filename': 'otif_pull.md', 'section_type': 'column_reference'}, page_content='## 📊 Column Names Reference\n\n**Format: column_name | description | synonymous_names**')]